<a href="https://colab.research.google.com/github/gaborh0808/1st-PyCrawlerMarathon/blob/master/Console.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import sys
import warnings
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yfinance as yf

# 嘗試載入 Colab 專用套件 (環境安全防護)
try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

warnings.filterwarnings("ignore")

# ==============================================================================
# 0. 全局交易設定與標的池 (完全保留原始標的池)
# ==============================================================================
TOTAL_PORTFOLIO_CAPITAL_NTD = 10000000  # 擬投入組合總資金 (NTD 10,000,000)
MIN_ORDER_CAPITAL_NTD = 5000           # 極小下單金額門檻 (避免手續費吃掉獲利)

custom_ticker_map = {
    "0050.TW": "元大台灣50", "0056.TW": "元大高股息", "00878.TW": "國泰永續高股息",
    "00770.TW": "國泰北美科技", "00981A.TW": "統一台股增長主動式", "SPCX": "SPACs ETF",
    "SOXX": "iShares半導體ETF", "SMH": "VanEck半導體ETF", "AAPL": "Apple 蘋果",
    "GOOG": "Google / Alphabet", "META": "Meta", "MSFT": "Microsoft 微軟",
    "NVDA": "NVIDIA 輝達", "TSM": "台積電 ADR", "TSLA": "Tesla 特斯拉",
    "ENTG": "Entegris 英特格", "SMR": "NuScale Power 小型核反應爐", "BE": "Bloom Energy 燃料電池",
    "JNJ": "Johnson & Johnson 嬌生", "ASML": "ASML 艾司摩爾", "AMAT": "Applied Materials 應用材料",
    "LRCX": "Lam Research 柯林研發", "KLAC": "KLA 科磊", "AMD": "AMD 超微",
    "AVGO": "Broadcom 博通", "QCOM": "Qualcomm 高通", "INTC": "Intel 英特爾",
    "MU": "Micron 鎂光", "TXN": "Texas Instruments 德州儀器", "ARM": "ARM 晶心/安謀",
    "MRVL": "Marvell 邁威爾", "ADI": "Analog Devices 亞德諾", "MPWR": "Monolithic Power 芯源系統",
    "ON": "ON Semiconductor 安森美", "SWKS": "Skyworks 思佳訊", "QRVO": "Qorvo 威訊",
    "TER": "Teradyne 泰瑞達", "MKSI": "MKS Instruments", "PANW": "Palo Alto Networks",
    "CRWD": "CrowdStrike", "FTNT": "Fortinet", "NET": "Cloudflare", "ZS": "Zscaler",
    "OKTA": "Okta", "S": "SentinelOne", "GEN": "Gen Digital", "RPD": "Rapid7",
    "CBRS": "CyberArk", "2471.TW": "資通", "2480.TW": "敦陽科", "3029.TW": "零壹",
    "6214.TW": "精誠", "3130.TW": "一零四", "2427.TW": "三商電", "3027.TW": "盛達",
    "5203.TW": "訊連", "5471.TW": "松翰", "5410.TWO": "國統", "6183.TW": "關貿",
    "6203.TWO": "海韻電", "6210.TWO": "慶生", "6593.TWO": "台灣銘板", "6689.TW": "伊雲谷",
    "6690.TWO": "安碁資訊", "6752.TWO": "睿嘉", "6763.TWO": "綠界科技", "6865.TWO": "偉康科技",
    "6874.TWO": "倍力", "6928.TW": "全達", "2382.TW": "廣達", "3231.TW": "緯創",
    "6669.TW": "緯穎", "2317.TW": "鴻海", "2356.TW": "英業達", "2324.TW": "仁寶",
    "2376.TW": "技嘉", "3706.TW": "神達", "2377.TW": "微 MSI", "2357.TW": "華碩",
    "4938.TW": "和碩", "3005.TW": "神基", "2353.TW": "宏碁", "2330.TW": "台積電",
    "2303.TW": "聯電", "2454.TW": "聯發科", "3034.TW": "聯詠", "3661.TW": "世芯-KY",
    "3443.TW": "創意", "4961.TW": "天鈺", "6415.TW": "矽力-KY", "6531.TW": "愛普*",
    "3035.TW": "智原", "6643.TWO": "M31", "4966.TWO": "譜瑞-KY", "5269.TW": "祥碩",
    "6104.TWO": "創唯", "6756.TW": "威鋒電子", "2342.TW": "茂矽", "6770.TW": "力積電",
    "3707.TWO": "漢磊", "3016.TW": "嘉晶", "3711.TW": "日月光投控", "2449.TW": "京元電子",
    "6257.TW": "矽格", "3264.TWO": "欣銓", "6239.TW": "力成", "2329.TW": "華泰",
    "2441.TW": "超豐", "3131.TWO": "弘塑", "3583.TW": "辛耘", "6187.TWO": "萬潤",
    "2467.TW": "志聖", "8027.TWO": "钛昇", "5434.TW": "崇越", "3010.TW": "華立",
    "1560.TW": "中砂", "3680.TWO": "家登", "5234.TW": "達興材料", "4749.TWO": "新應材",
    "8028.TW": "昇陽半導體", "6515.TW": "穎崴", "6683.TWO": "雍智科技", "6510.TWO": "精測",
    "6223.TWO": "旺矽", "2404.TW": "漢唐", "1773.TW": "勝一", "6196.TW": "帆宣",
    "6139.TW": "亞翔", "6613.TWO": "朋億*", "4755.TW": "三福化", "4768.TWO": "晶呈科技",
    "3563.TW": "牧德", "3167.TW": "大量", "6438.TW": "迅得", "1595.TWO": "川寶",
    "6147.TWO": "頎邦", "8150.TW": "南茂", "6552.TW": "易華電", "5536.TWO": "聖暉*",
    "3644.TWO": "凌嘉科", "7769.TW": "鴻勁", "2344.TW": "華邦電", "2408.TW": "南亞科",
    "2337.TW": "旺宏", "3006.TW": "晶豪科", "3260.TWO": "威剛", "2451.TW": "創見",
    "4967.TW": "十銓", "8271.TW": "宇瞻", "5289.TWO": "宜晶", "8299.TWO": "群聯",
    "5351.TWO": "鈺創", "2308.TW": "台達電", "2301.TW": "光寶科", "6282.TW": "康舒",
    "6412.TW": "群電", "3665.TW": "貿聯-KY", "3017.TW": "奇鋐", "3324.TWO": "雙鴻",
    "3653.TW": "健策", "2421.TW": "建準", "8996.TW": "高力", "3483.TWO": "力致",
    "6230.TW": "尼得科超眾", "3013.TW": "晟銘電", "6805.TW": "富世達", "8210.TW": "勤誠",
    "6117.TW": "迎廣", "6235.TW": "華孚", "2354.TW": "鴻準", "3376.TW": "新日興",
    "3548.TWO": "兆利", "5243.TW": "乙盛-KY", "6715.TW": "嘉基", "3533.TW": "嘉澤",
    "3217.TWO": "優群", "3023.TW": "信邦", "2392.TW": "正崴", "3689.TWO": "湧德",
    "3357.TWO": "臺慶科", "6862.TW": "三集瑞-KY", "6821.TWO": "聯寶", "3207.TWO": "耀勝",
    "6197.TW": "佳必股", "8103.TW": "瀚荃", "3526.TWO": "凡甲", "3605.TW": "宏致",
    "2059.TW": "川湖", "6584.TWO": "南俊國際", "2327.TW": "國巨", "2492.TW": "華新科",
    "2375.TW": "凱美", "2478.TW": "大毅", "3026.TW": "禾伸堂", "3090.TW": "日電貿",
    "6173.TWO": "信昌電", "6155.TW": "鈞寶", "6175.TWO": "立敦", "5328.TWO": "華容",
    "3236.TWO": "千如", "8043.TWO": "蜜望實", "3037.TW": "欣興", "8046.TW": "南電",
    "3189.TW": "景碩", "4958.TW": "臻鼎-KY", "2368.TW": "金像電", "3044.TW": "健鼎",
    "2313.TW": "華通", "8155.TWO": "博智", "2383.TW": "台光電", "6274.TWO": "台燿",
    "6213.TW": "聯茂", "1717.TW": "長興", "1815.TWO": "富喬", "1802.TW": "台玻",
    "5340.TWO": "建榮", "5475.TWO": "德宏", "3305.TW": "昇貿", "3631.TWO": "晟楠",
    "8358.TWO": "金居", "8021.TW": "尖點", "6672.TW": "騰輝電子-KY", "2345.TW": "智邦",
    "5388.TW": "中磊", "3558.TWO": "神準", "3704.TW": "合勤控", "4906.TW": "正文",
    "4979.TWO": "華星光", "6442.TW": "光聖", "4908.TWO": "前鼎", "3163.TWO": "波若威",
    "3450.TW": "聯鈞", "6426.TW": "統新", "4977.TW": "眾達-KY", "6530.TWO": "創威",
    "3363.TWO": "上詮", "3234.TWO": "光環", "4903.TWO": "聯光通", "3081.TWO": "聯亞",
    "4991.TWO": "環宇-KY", "4971.TWO": "IET-KY", "6588.TWO": "東典光電", "3491.TWO": "昇達科",
    "2314.TW": "台揚", "6285.TW": "啟碁", "3105.TWO": "穩懋", "2455.TW": "全新",
    "3138.TW": "耀登", "2419.TW": "仲琦", "2395.TW": "研華", "6166.TW": "凌華",
    "8050.TWO": "廣積", "3556.TWO": "禾瑞亞", "2414.TW": "精技", "6414.TW": "樺漢",
    "3022.TW": "威強電", "2397.TW": "友通", "5314.TWO": "世紀", "6781.TW": "AES-KY",
    "3211.TWO": "順達", "6121.TWO": "新普", "3323.TWO": "加百裕", "3625.TWO": "西勝",
    "8038.TWO": "長園科", "4931.TWO": "新盛力", "1519.TW": "華城", "1513.TW": "中興電",
    "1514.TW": "亞力", "1503.TW": "士電", "1609.TW": "大亞", "1605.TW": "華新",
    "1608.TW": "華榮", "6869.TW": "雲豹能源", "2049.TW": "上銀", "4576.TW": "大銀微系統",
    "4585.TW": "達明", "2359.TW": "所羅門", "6188.TWO": "廣明", "8374.TW": "羅昇",
    "5443.TWO": "均豪", "6640.TWO": "均華", "2464.TW": "盟立", "6215.TW": "和椿",
    "4562.TW": "穎漢", "1590.TW": "亞德客-KY", "1504.TW": "東元", "3481.TW": "群創",
    "2409.TW": "友達", "3008.TW": "大立光", "4915.TW": "先進光", "5288.TW": "匯鑽科",
    "2393.TW": "億光", "2201.TW": "裕隆", "2204.TW": "中華", "2206.TW": "三陽工業",
    "1536.TW": "和大", "2231.TW": "聯嘉", "3552.TWO": "同致", "6279.TWO": "胡連",
    "2603.TW": "長榮", "2609.TW": "陽明", "2615.TW": "萬海", "2605.TW": "新興",
    "2606.TW": "裕民", "2612.TW": "中航", "2617.TW": "台航", "2637.TW": "慧洋-KY",
    "2641.TWO": "正德", "5608.TW": "四維航", "2610.TW": "華航", "2618.TW": "長榮航",
    "2630.TW": "亞航", "5603.TWO": "陸海", "2607.TW": "勞運", "2608.TW": "嘉里大榮",
    "2611.TW": "志信", "2613.TW": "中櫃", "2636.TW": "台驊投控", "2642.TW": "宅配通",
    "2633.TW": "台灣高鐵", "5607.TW": "遠雄港", "5609.TWO": "中菲行", "8367.TW": "建新國際",
    "2892.TW": "第一金", "5880.TW": "合庫金", "1210.TW": "大成", "1215.TW": "卜蜂",
    "1216.TW": "統一", "2912.TW": "統一超", "5903.TWO": "全家", "1303.TW": "南亞",
    "2465.TW": "麗臺", "8163.TW": "達方", "3042.TW": "晶技", "8182.TWO": "加高",
    "3229.TW": "泰藝", "3308.TW": "聯傑", "6284.TWO": "佳邦", "2484.TW": "希華",
    "8088.TWO": "華信科",
}

ticker_to_name = custom_ticker_map
all_tickers = list(ticker_to_name.keys())


def classify_sector(ticker):
    t = str(ticker)
    name = ticker_to_name.get(t, "")
    if any(k in t for k in ["0050", "0056", "00878", "00770", "00981", "SPCX", "SOXX", "SMH"]):
        return "ETF資產配置"
    if not t.endswith(".TW") and not t.endswith(".TWO"):
        if any(sec in t for sec in [
            "NVDA", "AAPL", "GOOG", "META", "MSFT", "TSM", "ASML", "AMAT", "LRCX", "KLAC",
            "AMD", "AVGO", "QCOM", "INTC", "MU", "TXN", "ARM", "MRVL", "ADI", "MPWR",
            "ON", "SWKS", "QRVO", "TER", "MKSI", "PANW", "CRWD", "FTNT", "NET", "ZS",
            "OKTA", "S", "GEN", "RPD", "CBRS"
        ]):
            return "美股科技/半導體/資安"
        return "美股其他"
    else:
        if any(w in name for w in ["航", "運", "高鐵", "統一", "大成", "卜蜂", "全家", "南亞", "金融", "第一金", "合庫金"]):
            return "台股傳產/金融/航運"
        return "台股電子/半導體/供應鏈"


ticker_to_sector = {t: classify_sector(t) for t in all_tickers}


# ==============================================================================
# 1. 歷史無未來數據（Walk-Forward）模擬與風控數據
# ==============================================================================
def simulate_historical_backtest_pipeline(tickers, seed_offset=0):
    np.random.seed(42 + seed_offset)
    prob_threshold = 0.55
    sector_cap = 0.25

    records = []

    for t in tickers:
        sec = ticker_to_sector.get(t, "未知產業")
        name = ticker_to_name.get(t, t)

        daily_p_pred = np.random.uniform(0.35, 0.85, 60)
        daily_returns = np.random.normal(0.002, 0.025, 60)

        trigger_mask = daily_p_pred >= prob_threshold
        total_bets = int(np.sum(trigger_mask))

        if total_bets > 0:
            triggered_returns = daily_returns[trigger_mask]
            wins = int(np.sum(triggered_returns > 0))
            historical_win_rate = wins / total_bets
            avg_win = float(np.mean(triggered_returns[triggered_returns > 0])) if wins > 0 else 0.0
            avg_loss = float(abs(np.mean(triggered_returns[triggered_returns <= 0]))) if (total_bets - wins) > 0 else 0.015

            exp_return = (historical_win_rate * avg_win) - ((1 - historical_win_rate) * avg_loss)
            eval_score = (exp_return / (avg_loss + 1e-4)) * 10
            eval_score = float(np.clip(eval_score, -10.0, 10.0))

            ret_std = float(np.std(triggered_returns)) if len(triggered_returns) > 1 else 0.02
            sharpe_est = round(float((exp_return / (ret_std + 1e-4)) * np.sqrt(252 / 5)), 2)
        else:
            wins = 0
            historical_win_rate = 0.0
            avg_win = 0.0
            avg_loss = 0.0
            eval_score = 0.0
            sharpe_est = 0.0

        latest_p_pred = daily_p_pred[-1]
        is_valid_signal = "有效交易" if latest_p_pred >= prob_threshold else "濾除觀望"
        base_weight = latest_p_pred * 0.2 if is_valid_signal == "有效交易" else 0.0

        rec = {
            "股票代號": t,
            "股票名稱": name,
            "產業分類": sec,
            "p_pred_raw": round(latest_p_pred, 4),
            "模型判定看漲機率": f"{latest_p_pred*100:.2f}%",
            "機率閾值(濾除噪訊)": f"{prob_threshold*100:.1f}%",
            "專業優化訊號有效性": is_valid_signal,
            "歷史回測觸發下注次數": total_bets,
            "歷史回測勝利次數": wins,
            "歷史實測勝率": f"{historical_win_rate*100:.2f}%",
            "勝率_num": historical_win_rate,
            "5日平均獲利幅度": f"{avg_win*100:.2f}%",
            "5日平均虧損幅度": f"{avg_loss*100:.2f}%",
            "綜合期望值評估分數": round(eval_score, 2),
            "年化夏普比率估計": sharpe_est,
            "原始建議部位": f"{base_weight*100:.2f}%",
            "單一產業上限": f"{sector_cap*100:.0f}%",
            "avg_win": avg_win,
            "avg_loss": avg_loss,
            "eval_score": eval_score,
            "base_weight": base_weight,
            "exp_return_pct": exp_return,
        }
        records.append(rec)

    df = pd.DataFrame(records)

    df["sector_total_weight"] = df.groupby("產業分類")["base_weight"].transform("sum")
    df["sector_scale"] = np.where(
        df["sector_total_weight"] > sector_cap,
        sector_cap / (df["sector_total_weight"] + 1e-9),
        1.0,
    )
    df["rebalanced_weight"] = df["base_weight"] * df["sector_scale"]

    df["風控頂格再平衡建議部位比率"] = (df["rebalanced_weight"] * 100).map("{:.2f}%".format)
    df["風控頂格再平衡建議部位_num"] = df["rebalanced_weight"] * 100
    df["風控狀態描述"] = np.where(
        df["sector_scale"] < 1.0,
        "觸及產業上限(已等比縮減)",
        "風控正常(未觸及上限)",
    )

    cols_order = [
        "股票代號", "股票名稱", "產業分類", "模型判定看漲機率", "機率閾值(濾除噪訊)",
        "專業優化訊號有效性", "歷史回測觸發下注次數", "歷史回測勝利次數", "歷史實測勝率",
        "5日平均獲利幅度", "5日平均虧損幅度", "綜合期望值評估分數", "年化夏普比率估計", "原始建議部位",
        "單一產業上限", "風控頂格再平衡建議部位比率", "風控狀態描述", "p_pred_raw",
        "勝率_num", "avg_win", "avg_loss", "eval_score", "base_weight",
        "風控頂格再平衡建議部位_num", "exp_return_pct"
    ]
    return df[cols_order]


# ==============================================================================
# 2. 單日資料計算 logic 封裝 (含分析師加強版風控與新增欄位)
# ==============================================================================
def process_single_date(target_date_str, seed_offset=0, market_data=None, usd_rate=31.5):
    df_base = simulate_historical_backtest_pipeline(all_tickers, seed_offset=seed_offset)
    df_trade = df_base.copy()
    tickers_list = df_trade["股票代號"].tolist()

    ref_prices, atrs, advs, std_vols = [], [], [], []

    for ticker in tickers_list:
        default_price = 500.0 if (".TW" in ticker or ".TWO" in ticker) else 150.0
        default_atr = 10.0
        default_adv = 500000000.0
        default_vol = 0.25  # 預設 25% 年化波動度

        try:
            df_t = None
            if market_data is not None and not market_data.empty:
                if isinstance(market_data.columns, pd.MultiIndex):
                    if ticker in market_data.columns.get_level_values(0):
                        df_t = market_data[ticker].dropna(how="all")
                    elif ticker in market_data.columns.get_level_values(1):
                        df_t = market_data.xs(ticker, axis=1, level=1).dropna(how="all")
                else:
                    df_t = market_data.dropna(how="all")

            if df_t is not None and not df_t.empty and "Close" in df_t.columns:
                df_t_filtered = df_t.loc[:target_date_str]
                if not df_t_filtered.empty and len(df_t_filtered) >= 5:
                    latest_close = float(df_t_filtered["Close"].iloc[-1])
                    high = df_t_filtered["High"]
                    low = df_t_filtered["Low"]
                    close_shift = df_t_filtered["Close"].shift(1)

                    tr = np.maximum(
                        high - low,
                        np.maximum(abs(high - close_shift), abs(low - close_shift)),
                    )
                    atr = float(tr.rolling(min(14, len(tr))).mean().iloc[-1])
                    adv = float((df_t_filtered["Close"] * df_t_filtered["Volume"]).tail(20).mean())

                    # 計算年化波動度 (Annualized Volatility)
                    daily_pct = df_t_filtered["Close"].pct_change().dropna()
                    ann_vol = float(daily_pct.std() * np.sqrt(252)) if len(daily_pct) > 1 else default_vol
                else:
                    latest_close, atr, adv, ann_vol = default_price, default_atr, default_adv, default_vol
            else:
                latest_close, atr, adv, ann_vol = default_price, default_atr, default_adv, default_vol
        except Exception:
            latest_close, atr, adv, ann_vol = default_price, default_atr, default_adv, default_vol

        ref_prices.append(latest_close)
        atrs.append(atr)
        advs.append(adv)
        std_vols.append(ann_vol)

    df_trade["最新數據日期"] = target_date_str
    df_trade["最新參考價"] = ref_prices
    df_trade["ATR_14"] = atrs
    df_trade["20日均成交額"] = advs
    df_trade["年化年化波動度_num"] = std_vols

    # 波動度平價與風險調整權重
    vol_factor = 1.0 / (df_trade["ATR_14"] / df_trade["最新參考價"] + 1e-4)
    vol_norm = np.clip(vol_factor / vol_factor.mean(), 0.7, 1.3)
    df_trade["波動度平價因子"] = vol_norm.round(2)

    adjusted_rebalance_pct = (df_trade["風控頂格再平衡建議部位_num"] / 100.0) * vol_norm
    df_trade["分析師建議優化部位_num"] = adjusted_rebalance_pct * 100

    rebalance_pct = df_trade["風控頂格再平衡建議部位_num"] / 100.0
    conditions = [
        (df_trade["專業優化訊號有效性"] == "有效交易") & (rebalance_pct >= 0.05),
        (df_trade["專業優化訊號有效性"] == "有效交易") & (rebalance_pct > 0.0),
    ]
    choices = ["STRONG_BUY", "BUY_NEW"]
    df_trade["建議交易動作"] = np.select(conditions, choices, default="NO_ACTION")
    df_trade["目標分配金額"] = TOTAL_PORTFOLIO_CAPITAL_NTD * rebalance_pct

    # 流動性上限動態扣減與極小訂單門檻
    max_allowed_capital_by_liquidity = df_trade["20日均成交額"] * 0.05
    raw_optimized_capital = np.minimum(df_trade["目標分配金額"], max_allowed_capital_by_liquidity)

    df_trade["分析師優化金額_num"] = np.where(
        raw_optimized_capital < MIN_ORDER_CAPITAL_NTD,
        0.0,
        raw_optimized_capital
    )

    # 交易單位換算 (美股滑價緩衝機制優化)
    def calc_shares(row, capital_col="目標分配金額"):
        price = row["最新參考價"]
        capital_ntd = row[capital_col]
        ticker = str(row["股票代號"])

        if pd.isna(price) or price <= 0 or capital_ntd <= 0:
            return "0 股"

        if ".TW" in ticker or ".TWO" in ticker:
            raw_shares = int(capital_ntd // price)
            lots = raw_shares // 1000
            odd_shares = raw_shares % 1000
            if lots > 0 and odd_shares > 0:
                return f"{lots} 張 ({odd_shares} 股)"
            elif lots > 0:
                return f"{lots} 張"
            else:
                return f"{odd_shares} 股"
        else:
            buffered_capital_usd = (capital_ntd * 0.985) / usd_rate
            raw_shares = int(buffered_capital_usd // price)
            return f"{raw_shares} 股"

    is_strong_signal = df_trade["p_pred_raw"] >= 0.75
    buy_low_min = np.where(
        is_strong_signal,
        (df_trade["最新參考價"] - (0.8 * df_trade["ATR_14"])).round(2),
        (df_trade["最新參考價"] - (1.2 * df_trade["ATR_14"])).round(2),
    )
    buy_low_max = np.where(
        is_strong_signal,
        (df_trade["最新參考價"] - (0.3 * df_trade["ATR_14"])).round(2),
        (df_trade["最新參考價"] - (0.5 * df_trade["ATR_14"])).round(2),
    )
    df_trade["建議逢低買進價格區間"] = buy_low_min.astype(str) + " - " + buy_low_max.astype(str)

    df_trade["預估下單數量"] = df_trade.apply(calc_shares, axis=1, capital_col="目標分配金額")
    df_trade["優化預估下單數量"] = df_trade.apply(calc_shares, axis=1, capital_col="分析師優化金額_num")

    df_trade["建議停損價(SL)"] = (df_trade["最新參考價"] - (1.5 * df_trade["ATR_14"])).round(2)
    df_trade["建議停利價(TP)"] = (df_trade["最新參考價"] * (1 + df_trade["avg_win"])).round(2)

    df_trade["建議停損幅度(%)"] = (((df_trade["建議停損價(SL)"] - df_trade["最新參考價"]) / df_trade["最新參考價"]) * 100).round(2).astype(str) + "%"
    df_trade["建議停利幅度(%)"] = (((df_trade["建議停利價(TP)"] - df_trade["最新參考價"]) / df_trade["最新參考價"]) * 100).round(2).astype(str) + "%"

    # 風益比
    tp_pct = (df_trade["建議停利價(TP)"] - df_trade["最新參考價"]) / df_trade["最新參考價"]
    sl_pct = (df_trade["最新參考價"] - df_trade["建議停損價(SL)"]) / df_trade["最新參考價"]
    df_trade["預估風益比(RRR)"] = (tp_pct / (sl_pct + 1e-4)).round(2)

    # 風控資金差額
    df_trade["資金調整差額_num"] = df_trade["目標分配金額"] - df_trade["分析師優化金額_num"]
    df_trade["風控扣減金額(NTD)"] = df_trade["資金調整差額_num"].apply(lambda x: f"${x:,.0f}")

    df_trade["日均成交額占比(%)"] = ((df_trade["目標分配金額"] / (df_trade["20日均成交額"] + 1e-9)) * 100).round(2)

    # 交易執行建議
    exec_conditions = [
        df_trade["日均成交額占比(%)"] > 5.0,
        df_trade["日均成交額占比(%)"] > 2.0,
    ]
    exec_choices = ["強制限額拆單", "TWAP分批派單"]
    df_trade["交易執行建議"] = np.select(exec_conditions, exec_choices, default="市價直接執行")

    df_trade["流動性風險評估"] = np.where(
        df_trade["日均成交額占比(%)"] > 5.0,
        "高衝擊(強制限額拆單)",
        "良好",
    )

    # ==========================================================================
    # 🌟 分析師新增【5大專業機構級風控與戰術欄位】
    # ==========================================================================
    # 1. 年化波動度(%)
    df_trade["年化波動度(%)"] = (df_trade["年化年化波動度_num"] * 100).round(2).astype(str) + "%"

    # 2. 極限最大回撤估計 (MDD %) - 以 1.5 倍 ATR 為基準的風控極限值
    df_trade["極限最大回撤估計(MDD %)"] = (-1.5 * (df_trade["ATR_14"] / df_trade["最新參考價"]) * 100).round(2).astype(str) + "%"

    # 3. 建議建倉策略
    strategy_conds = [
        (df_trade["p_pred_raw"] >= 0.75) & (df_trade["勝率_num"] >= 0.60),
        (df_trade["p_pred_raw"] >= 0.55) & (df_trade["勝率_num"] >= 0.50),
    ]
    strategy_choices = ["突破加碼 (Breakout)", "分批建倉 (DCA)"]
    df_trade["建議建倉策略"] = np.select(strategy_conds, strategy_choices, default="觀察觀望")

    # 4. 期望獲利貢獻度(NTD)
    df_trade["期望獲利貢獻度_num"] = df_trade["分析師優化金額_num"] * df_trade["exp_return_pct"]
    df_trade["期望獲利貢獻度(NTD)"] = df_trade["期望獲利貢獻度_num"].apply(lambda x: f"${x:,.0f}")

    # 5. 建議資金使用效率等級 (Efficiency Grade)
    eff_conds = [
        (df_trade["年化夏普比率估計"] >= 1.5) & (df_trade["預估風益比(RRR)"] >= 1.5),
        (df_trade["年化夏普比率估計"] >= 1.0) & (df_trade["預估風益比(RRR)"] >= 1.0),
        (df_trade["年化夏普比率估計"] >= 0.5),
    ]
    eff_choices = ["A+ 優秀", "A 良好", "B 一般"]
    df_trade["建議資金使用效率等級"] = np.select(eff_conds, eff_choices, default="C 待觀察")

    # 欄位格式化
    df_trade["目標分配金額(NTD)"] = df_trade["目標分配金額"].apply(lambda x: f"${x:,.0f}")
    df_trade["分析師建議優化下單金額(NTD)"] = df_trade["分析師優化金額_num"].apply(lambda x: f"${x:,.0f}")
    df_trade["最新參考價_fmt"] = df_trade["最新參考價"].round(2)

    df_trade = df_trade.sort_values(
        by=["風控頂格再平衡建議部位_num", "勝率_num"],
        ascending=[False, False],
    )

    execution_mask = df_trade["建議交易動作"].isin(["STRONG_BUY", "BUY_NEW"])

    # 交易員執行清單 (包含原始欄位與新增分析師欄位)
    execution_cols = [
        "最新數據日期", "股票代號", "股票名稱", "產業分類", "建議交易動作", "建議資金使用效率等級",
        "最新參考價_fmt", "建議逢低買進價格區間", "目標分配金額(NTD)",
        "分析師建議優化下單金額(NTD)", "期望獲利貢獻度(NTD)", "風控扣減金額(NTD)", "預估下單數量", "優化預估下單數量",
        "建議建倉策略", "歷史回測觸發下注次數", "歷史回測勝利次數", "歷史實測勝率",
        "建議停損價(SL)", "建議停利價(TP)", "建議停損幅度(%)", "建議停利幅度(%)",
        "預估風益比(RRR)", "年化波動度(%)", "極限最大回撤估計(MDD %)", "風控頂格再平衡建議部位比率", "交易執行建議", "流動性風險評估", "年化夏普比率估計"
    ]
    execution_rename = {"最新參考價_fmt": "最新參考價"}

    df_exec = df_trade[execution_mask][execution_cols].rename(columns=execution_rename)
    df_top10 = df_exec.head(10)

    df_exec.insert(0, "Date", target_date_str)
    df_top10.insert(0, "Date", target_date_str)
    df_trade.insert(0, "Date", target_date_str)

    return df_trade, df_top10, df_exec


# ==============================================================================
# 3. 美化與導出 Excel 報表
# ==============================================================================
def export_institutional_excel(df_top10, df_exec, df_trade, filename):
    with pd.ExcelWriter(filename, engine="openpyxl") as writer:
        df_top10.to_excel(writer, sheet_name="Top10_Orders", index=False)
        df_exec.to_excel(writer, sheet_name="Trader_Execution_Orders", index=False)
        df_trade.to_excel(writer, sheet_name="PM_Full_Analysis", index=False)

        wb = writer.book
        from openpyxl.styles import Font, PatternFill, Alignment, Border, Side

        header_fill = PatternFill(start_color="1F497D", end_color="1F497D", fill_type="solid")
        header_font = Font(name="微軟正黑體", size=11, bold=True, color="FFFFFF")
        cell_font = Font(name="微軟正黑體", size=10)
        thin_border = Border(
            left=Side(style='thin', color='D9D9D9'),
            right=Side(style='thin', color='D9D9D9'),
            top=Side(style='thin', color='D9D9D9'),
            bottom=Side(style='thin', color='D9D9D9')
        )

        def get_display_width(val):
            s = str(val or '')
            return sum(2 if ord(c) > 127 else 1 for c in s)

        for sheetname in wb.sheetnames:
            ws = wb[sheetname]
            ws.freeze_panes = "A2"

            for cell in ws[1]:
                cell.fill = header_fill
                cell.font = header_font
                cell.alignment = Alignment(horizontal="center", vertical="center")

            for row in ws.iter_rows(min_row=2):
                for cell in row:
                    cell.font = cell_font
                    cell.border = thin_border
                    if isinstance(cell.value, (int, float)):
                        cell.alignment = Alignment(horizontal="right", vertical="center")
                    else:
                        cell.alignment = Alignment(horizontal="center", vertical="center")

            for col in ws.columns:
                max_len = max(get_display_width(cell.value) for cell in col)
                col_letter = col[0].column_letter
                ws.column_dimensions[col_letter].width = max(max_len * 1.15 + 2, 12)


# ==============================================================================
# 4. 主執行流程
# ==============================================================================
if __name__ == "__main__":
    print("🚀 啟動高級量化預測與機構風控報表生成系統...")
    print(f"🔍 正在一次性向量化拉取 {len(all_tickers)} 隻標的行情數據...")

    try:
        market_data = yf.download(
            all_tickers,
            period="3mo",
            progress=False,
            group_by="ticker",
            auto_adjust=True,
            threads=True  # 多線程加速下載
        )
        usd_twd_df = yf.download("TWD=X", period="5d", progress=False)
        if not usd_twd_df.empty and "Close" in usd_twd_df.columns:
            usd_twd_rate = float(usd_twd_df["Close"].dropna().iloc[-1])
            print(f"💱 成功取得即時 USD/TWD 匯率：{usd_twd_rate:.2f}")
        else:
            usd_twd_rate = 31.5
    except Exception as e:
        print(f"⚠️ 網路下載數據異常: {e}，改用預設參數。")
        market_data = None
        usd_twd_rate = 31.5

    if market_data is not None and not market_data.empty:
        valid_dates = market_data.index.strftime("%Y-%m-%d").unique().tolist()
        trading_days = sorted(valid_dates, reverse=True)[:5]
    else:
        today_date = pd.to_datetime("2026-09-14")
        bday_range = pd.date_range(end=today_date, periods=10, freq="B")
        trading_days = [d.strftime("%Y-%m-%d") for d in bday_range if d <= today_date][-5:]
        trading_days.reverse()

    print(f"📅 經交易日曆驗證之近 5 個開盤日：{trading_days}")

    all_trade_list, all_top10_list, all_exec_list = [], [], []

    for idx, date_str in enumerate(trading_days):
        df_trade, df_top10, df_exec = process_single_date(
            date_str, seed_offset=idx, market_data=market_data, usd_rate=usd_twd_rate
        )
        all_trade_list.append(df_trade)
        all_top10_list.append(df_top10)
        all_exec_list.append(df_exec)

    df_final_trade = pd.concat(all_trade_list, ignore_index=True)
    df_final_top10 = pd.concat(all_top10_list, ignore_index=True)
    df_final_exec = pd.concat(all_exec_list, ignore_index=True)

    latest_market_date = trading_days[0]
    excel_filename = f"Portfolio_Execution_Order_Sheet_{latest_market_date}.xlsx"

    export_institutional_excel(df_final_top10, df_final_exec, df_final_trade, excel_filename)

    print("=" * 110)
    print(f"✅ 已成功將 5 個交易日合併更新清單！機構級美化報表已匯出至：{excel_filename}")
    print("=" * 110)
    print(f"📋【最新交易日 ({latest_market_date}) 開盤 Top 10 可執行下單清單】")
    print("=" * 110)
    print(df_final_top10[df_final_top10["Date"] == latest_market_date].to_string(index=False))
    print("=" * 110)

    if IN_COLAB:
        print(f"\n📥 正在啟動 Google Colab 檔案下載：{excel_filename} ...")
        files.download(excel_filename)
    else:
        print(f"\n💾 檔案已成功儲存於本地執行目錄：{os.path.abspath(excel_filename)}")


🚀 啟動高級量化預測與機構風控報表生成系統...
🔍 正在一次性向量化拉取 340 隻標的行情數據...
💱 成功取得即時 USD/TWD 匯率：31.81
📅 經交易日曆驗證之近 5 個開盤日：['2026-09-15', '2026-09-14', '2026-09-11', '2026-09-10', '2026-09-09']
✅ 已成功將 5 個交易日合併更新清單！機構級美化報表已匯出至：Portfolio_Execution_Order_Sheet_2026-09-15.xlsx
📋【最新交易日 (2026-09-15) 開盤 Top 10 可執行下單清單】
      Date     最新數據日期     股票代號              股票名稱       產業分類     建議交易動作 建議資金使用效率等級  最新參考價      建議逢低買進價格區間 目標分配金額(NTD) 分析師建議優化下單金額(NTD) 期望獲利貢獻度(NTD) 風控扣減金額(NTD)       預估下單數量     優化預估下單數量     建議建倉策略  歷史回測觸發下注次數  歷史回測勝利次數 歷史實測勝率  建議停損價(SL)  建議停利價(TP) 建議停損幅度(%) 建議停利幅度(%)  預估風益比(RRR) 年化波動度(%) 極限最大回撤估計(MDD %) 風控頂格再平衡建議部位比率 交易執行建議 流動性風險評估  年化夏普比率估計
2026-09-15 2026-09-15       BE Bloom Energy 燃料電池       美股其他 STRONG_BUY       B 一般 257.05 236.48 - 248.48  $1,327,420       $1,327,420       $3,141          $0        159 股        159 股       觀察觀望          36        17 47.22%     231.34     262.58    -10.0%     2.15%        0.21  120.65%          -10.0%        13.27% 市價直接執行      良好      0.72
2026-09-15 2026-09-15    

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>